## 12/06/2026 - Scrape two pages + clean data [Claude Code]

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import requests
import json

In [ ]:
homepage_url = 'https://quotes.toscrape.com/js/' # The URL of the homepage of the quotes website
res = requests.get(homepage_url) # Send a GET request to the homepage URL
print(res.status_code) # Print the status code of the response to check if the request was successful (200 means OK)
soup = BeautifulSoup(res.text, 'html.parser') # Parse the HTML content of the homepage using BeautifulSoup

In [ ]:
all_quotes = []
base_url = 'https://quotes.toscrape.com/js' # Base URL to resolve relative links 

current_page_url = 'https://quotes.toscrape.com/js/page/1/'

page_count = 0

while current_page_url: 
    
    print(f"Scraping page: {current_page_url}") # Print the current page URL being scraped

    res = requests.get(current_page_url) # Send a GET request to the current page URL

    soup = BeautifulSoup(res.text, 'html.parser') # Parse the HTML content of the current page using BeautifulSoup

    next_button = soup.select_one('li.next a') 

    scripts = soup.select('script') # Select all script tags from the page

    quote_data = []
    for script in scripts:
        if 'var data =' in script.text: # Check if the script tag contains the variable declaration for the quotes data
            vardata = script.text # Get the text content of the script tag
            start = vardata.find('var data =') + len('var data =') # Find the start position of the data
            end = vardata.find('];', start) + 1 # Find the end position of the data
            data_str = vardata[start:end] # Extract the JSON string containing the quotes data 
            quotes = json.loads(data_str) # Parse the JSON string into a Python list of dictionaries
            quote_data.extend(quotes) # Add the quotes from the current page to the overall list of quotes 

    df = pd.DataFrame(quote_data) # Create a DataFrame from the list of quotes
    all_quotes.append(df)

    if next_button: # Check if the next button exists before trying to access its href attribute
        current_page_url = urljoin(base_url, next_button.get('href')) # Construct the full URL for the next page
    else:
        current_page_url = None # If there is no next button, exit the loop

    page_count += 1 # Increment the page count
    if page_count == 2:
        break

all_quotes_df = pd.concat(all_quotes, ignore_index=True) # Concatenate all DataFrames into one; ignore_index avoids duplicate indices
print(f"\nRaw shape: {all_quotes_df.shape}")
all_quotes_df.head()

## Data Cleaning

The raw DataFrame has three columns that need fixing:
- **`author`** — nested dict; extract `name`, `goodreads_link`, and `slug` into separate columns
- **`tags`** — Python list; join into a comma-separated string
- **`text`** — wrapped in curly unicode quotes (`\u201c` / `\u201d`); strip them off

In [ ]:
# Inspect the raw DataFrame before cleaning
all_quotes_df.info()
print()
print(all_quotes_df.iloc[0]) # Show the first row to see the nested structure

In [ ]:
# --- Flatten the nested 'author' dict into separate columns ---
all_quotes_df['author_name'] = all_quotes_df['author'].apply(lambda x: x['name'])
all_quotes_df['author_goodreads_link'] = all_quotes_df['author'].apply(lambda x: x['goodreads_link'])
all_quotes_df['author_slug'] = all_quotes_df['author'].apply(lambda x: x['slug'])

# --- Join the 'tags' list into a comma-separated string ---
all_quotes_df['tags'] = all_quotes_df['tags'].apply(lambda x: ', '.join(x))

# --- Strip the curly opening/closing quotes from the 'text' column ---
all_quotes_df['text'] = all_quotes_df['text'].str.strip('\u201c\u201d')

# --- Drop the original nested 'author' column ---
all_quotes_df = all_quotes_df.drop(columns=['author'])

# --- Reorder columns so text and author_name come first ---
all_quotes_df = all_quotes_df[['text', 'author_name', 'author_goodreads_link', 'author_slug', 'tags']]

print(f"Cleaned shape: {all_quotes_df.shape}")
all_quotes_df.head()

In [ ]:
all_quotes_df.to_csv('twopages_quotes_js_clean.csv', index=False)
print(f"Saved {len(all_quotes_df)} rows to twopages_quotes_js_clean.csv")